# Customer churn 

Customer churn is a problem that affects all industries.
Often a lot of energy is put into acquiring customers, but retaining them is neglected.
With a customer churn forecast, those customers who are likely to churn can be identified.
The company can then perhaps win them back with special offers or other attention.
Data pre-processing for classification is discussed as part of the exercise. 
The decision tree as a classifier is repeated and deepened from the first half of the semester. 
At the end of the exercise, you will be able to carry out a quality assessment of the model for typical classification problems and evaluate an existing ML solution in terms of its suitability for the problem.

The Jupyter Notebook depends on the scikit-learn version, among other things.
Please make sure that your version is up to date:

In [ ]:
import sklearn
sklearn.__version__

An overview of the current version can be found at https://scikit-learn.org/dev/versions.html.
This notebook is designed for version 1.2.

## Task 1: Creating the basic framework

### Task 1.1: Importing modules

<span style="color:blue">
 As a first step, load the libraries for analyzing and manipulating data presented in the lecture and first exercise, as well as scikit-learn.
</span>

In [ ]:
import scipy

Now we download the file `CustomerChurn.xlsx`.
The code for this is already provided.
IBM presents the data set
[in a blog post](https://community.ibm.com/community/user/businessanalytics/blogs/steven-macko/2019/07/11/telco-customer-churn-1113)
before.

In [ ]:
import os
import requests
import shutil
import numpy as np
import pandas as pd

file_name = "CustomerChurn.xlsx"
if file_name not in os.listdir("."):
    print(f"Download file '{file_name}'")
    r = requests.get("https://public.dhe.ibm.com/software/data/sw-library/cognos/mobile/C11/data/CustomerChurn.xlsx", stream=True)
    if r.status_code == 200:
        with open(file_name, 'wb') as f:
            r.raw.decode_content = True
            shutil.copyfileobj(r.raw, f)
else:
   print(f"File '{file_name}' is already downloaded")

print("Contents of folder:")
for file_entry in os.listdir("."):
    print("- ", file_entry)

#### Task 1.2: Reading the data set

In task 1.2, the data required for the analyses should be read in.
The module `pandas` provides the option of reading Excel files with the function `read_excel()`. 
You can find more information at the following link:
https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_excel.html

<span style="color:blue">Load the file `CustomerChurn.xlsx` using the procedure shown in the previous exercise.
 Save the data in a DataFrame called `df` and have it visualized.
</span> 

In [ ]:
df = pd.read_excel(file_name)
df.head() 

#### Task 1.3: Output of individual columns

By specifying the column name, various data from the dataframe can be output.
As an example, the following statement outputs the rows of the columns `Senior Citizen` and `Paperless Billing`.

```python
df[['Senior Citizen', 'Paperless Billing']]
```

<span style="color:blue">Output the rows of the `Total Charges`, `Churn` and `Online Security` columns of the DataFrame.
</span> 

In [ ]:
df[['Total Charges', 'Churn', 'Online Security']].head()


## Task 2: Data preprocessing <a class="anchor" id="second-task"></a>

In task 2, the data required for the analyses should be preprocessed.

#### Task 2.1: Determining the scales of the columns

In the lecture, various scales for attributes were presented.
The method `df.info()` shows which type has been determined within pandas.
A short technical explanation of the types is available
[in Manual](https://pandas.pydata.org/docs/user_guide/basics.html#dtypes).

<span style="color:blue">Execute the following statement and interpret the result.
 Which attribute is currently being measured on which scale?
 Which scale should the respective attribute actually have?
</span>

In [ ]:
df.info()

#### Task 2.2: Conversion to correct types

Now we want to extract numerical values ​​from the text in the `Total Charges` column that we can use further.

In [ ]:
df["Total Charges"] = pd.to_numeric(df['Total Charges'], errors='coerce')
df

<span style="color:blue">(a) Execute the following statement and interpret the result.
 What has changed?
</span>

In [ ]:
df.info()

Spoiler alert: Later in the Jupyter Notebook we want to use machine learning.
That's why we want to convert all ‘Yes’ entries into ones and all ‘No’ entries into zeros.
So that you can safely call the code multiple times, conversions that have already taken place are retained (1 remains 1 and 0 remains 0).
Here is the example code snippet:

```python
df['MY_ATTRIBUTE'] = df['MY_ATTRIBUTE'].map({'Yes': 1, 'No': 0, 1: 1, 0: 0})
```

<span style="color:blue">(b) Convert all "Yes" and "No" entries for all columns.
 Save the new version back in the DataFrame `df`.
</span>

In [ ]:
# Define the mapping
yes_no_map = {'Yes': 1, 'No': 0, 1: 1, 0: 0}

# Apply mapping to all columns where applicable
df = df.map(lambda x: yes_no_map.get(x, x))

Converting the contract types and the payment method is more complex.
That's why we'll skip that for this exercise and just delete the values.

In [ ]:
df.drop(columns=["Payment Method", "Contract", "Internet Service"], inplace=True)
df

## Task 3: Analysis of the data set <a class="anchor" id="third-task"></a>

After saving the data in a `DataFrame` and preprocessing it, it can be analyzed. 

### Task 3.1: Basic probability that a customer will churn

The `pandas` module provides a number of descriptive analysis tools, e.g. calculating means, variances and standard deviations of individual columns.
For more information, see the following documentation:
https://pandas.pydata.org/pandas-docs/stable/reference/frame.html#computations-descriptive-stats.

For example, the following statement outputs the probability that a customer will churn.

<span style="color:blue">(a) Execute the following statement and interpret the result.
</span>

In [ ]:
df['Churn'].describe()

<span style="color:blue"> (b) Call the `.describe()` method of the DataFrame.
 Interpret the results.
</span>

In [ ]:
df.describe()


### Task 3.2: Probability of churn for different groups

<span style="color:blue">Divide the customers into self-selected, sensible groups.
 Give the mean value for each group and interpret the result.
</span>

An alternative to the `df.groupby()` method is to pre-filter the columns yourself.
This looks easier at first, but you have to be careful while working!

```python
print("Senior Citizen - Yes")
display(df[df['Senior Citizen'] == 'Yes'].mean())
print()
print("Senior Citizen - No")
display(df[df['Senior Citizen'] == 'No'].mean())
```

Or if "Yes" has already been converted to a 1 and "No" to a 0:

```python
print("Senior Citizen - Yes")
display(df[df['Senior Citizen'] == 1].mean())
print()
print("Senior Citizen - No")
display(df[df['Senior Citizen'] == 0].mean())
```

By comparing a Pandas column with a specific value, a so-called mask is created. 
A mask is a list (in this case a `pd.Series`) of Boolean values.
This list refers to a DataFrame and has just as many entries.
The entry at the `i`-th position in the mask refers to the entry at the same position in the DataFrame.
If the value "TRUE" is in the mask at the `i`-th position, it means that the corresponding value is selected from the DataFrame.
If the value "FALSE" is in the mask, the value from the DataFrame is ignored.
Masks can easily be created yourself using a comparison operator and a `pd.Series`.

The first mask is created using the expression `df['Senior Citizen'] == 'Yes'` or `df['Senior Citizen'] == 1` generated.
For more information about masks, see https://www.python-kurs.eu/python_numpy_maskierung.php.

<span style="color:blue">Use a mask to specify all customers who pay less than $35 per month.
 How many are there?
</span>

In [ ]:
# Create a mask for customers who pay less than $35 per month
low_cost_mask = df['Monthly Charges'] < 35

# Apply the mask to the DataFrame
low_cost_customers = df[low_cost_mask]

# Output the number of such customers
print(f"Number of customers who pay less than $35/month: {len(low_cost_customers)}")

### Task 3.3: Fee payment by different groups

Within this task, the fee structure for each customer group is to be examined in more detail.
Boxplots are a graphical tool frequently used in statistics to visualize the location and frequency distributions of parameters.
In addition to the median, the minima and maxima as well as the 1st and 3rd quartiles are visually displayed.
For more information on boxplots and their interpretation, see the following link:
https://www.crashkurs-statistik.de/boxplots/ 
`pandas` offers the method `df.boxplot()` provides the ability to create boxplots for columns:
https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.boxplot.html.

The following method creates an example boxplot for the monthly fee. 

In [ ]:
import matplotlib.pyplot as plt
df.boxplot(column='Monthly Charges')
plt.show()

<span style="color:blue"> (a) Copy the code and modify it so that a boxplot is displayed for the current contract term and one for the total amounts billed so far.
</span>

In [ ]:
# Your Code...

In task 3.2, you used the `df.groupby` method to determine the empirical probability of migration of the various self-defined groups.

<span style="color:blue">(b) Identify whether there are differences in the monthly costs for the groups you selected above.
If you have chosen exactly the group as in the example, please change your group.
</span>

In [ ]:
df.boxplot(
    column="Monthly Charges",
    by=['Partner', 'Dependents']  # enter your attributes for group classification selected above here
)
plt.title("")
plt.show()

## Task 4: An ML-based forecast

With the first analysis, some fairly simple conclusions could be drawn from the data. 
In task 4, an algorithm is to be developed that predicts whether a certain customer would churn.
This is where machine learning comes in:
A program is developed that learns from the sample data whether a certain customer would churn.

### Task 4: 1 Initialization and training of the decision tree

For machine learning, the data is divided into two arrays in a first step:
`X` contains all the values ​​from which the relationship is to be learned,
and `y` contains only the value of the `Churn` column for this row.
The classification algorithms compare the attribute values ​​of `X` with the corresponding values ​​of `y`,
to identify patterns of how different attribute values ​​predict customer churn

In [ ]:
X = df.drop(
    ['Customer ID', 'LoyaltyID', 'Churn'],  # Name all columns that must not be used for learning
    axis=1
).values
y = df['Churn'].values  # The goal to be learned



Finally, the `X` and `y` arrays are separated into two parts, a training set and a test set.
As discussed in the lecture, the training set is fed into the algorithm to form a trained model.
Once the model is trained, it is used to classify the test set so that the quality of the model can be determined.

Training and test data can be generated from the matrix `X` and the vector `y` using cross-validation.
This is implemented by the function `train_test_split` from the sklearn module
(http://scikit-learn.org/stable/modules/cross_validation.html).
The following code generates 80% training and 20% test data and stores them in new arrays.
The parameter `random_state=42` ensures that the same data is selected as test and training values ​​each time.

In [ ]:
from sklearn import model_selection
X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=0.2, random_state=42)

In this exercise, a decision tree is used to classify whether a customer is about to leave the company or not.
To create this tree, an instance of an untrained decision tree classifier is first initialized
(Here, the maximum depth of the tree is initially set to 3).

In [ ]:
from sklearn import tree
clf_dt = tree.DecisionTreeClassifier(max_depth=3)

In a next step, the decision tree is trained using the training data using the ```fit()``` method (http://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html#sklearn.tree.DecisionTreeClassifier.fit).

In [ ]:
clf_dt.fit(X_train, y_train)

The scikit-learn decision tree not only needs numerical values ​​everywhere, but also no data can be missing.
That's why we simply discard all rows in which we have missing values.

In [ ]:
df.dropna(axis=1, how='any', inplace=True)
df

Now we call the learning process again:

In [ ]:
X = df.drop(['Customer ID', 'LoyaltyID', 'Churn'], axis=1).values
y = df['Churn'].values
X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=0.2, random_state=42)

clf_dt.fit(X_train, y_train)

Now that the decision tree is ready, its quality can be assessed using the test data.
One way to check the model is the `.score()` method,
which determines the average accuracy (_Accuracy_) of the given test data in classification problems.

Further information on the `score()` method can be found at the following link:
http://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html#sklearn.tree.DecisionTreeClassifier.score

In [ ]:
clf_dt.score(X_test, y_test)

<span style="color:blue">
 Put the result in the context of what we already know based on the arithmetic mean alone.
 Would you use this model in practice?
</span>

In [ ]:
print("Probability that the customer will churn", df['Churn'].mean() * 100, "%")
print("Probability that the customer will not churn", (1 - df['Churn'].mean()) * 100, "%")

### Task 4.2: Predicting the likelihood of churn for specific customers

After the decision tree has been trained and the accuracy (_Accuracy_) has been evaluated, the next step is to analyze
whether specific customers would churn.

    SeniorCitizen
    Partner
    Dependents
    Tenure
    PhoneService
    PaperlessBilling
    MonthlyCharges
    ...
    

<span style="color:blue"> (a) Parameterize the variables with values ​​of your choice and interpret the result.
</span>

In [ ]:
# Your adjustments:

SeniorCitizen = 0
Partner = 0
Dependents = 0
Tenure = 0
PhoneService = 0
PaperlessBilling = 0
MonthlyCharges = 0
...

The previously parameterized variables are merged into an array and the trained decision tree is used to predict whether a customer will churn. 

<span style="color:blue"> (b) Run the following code and interpret the input and output.
</span>

In [ ]:
customer = np.array(
    [SeniorCitizen, Partner, Dependents, Tenure, PhoneService, PaperlessBilling, MonthlyCharges]
).reshape(1, -1)
pred = clf_dt.predict(customer)

if pred[0] == 0:  # No -> 0
    print("The customer stays")
elif pred[0] == 1:  # Yes -> 1
    print("The customer moves away")   

In addition to predicting the binary variable `Churn`, the probabilities of belonging to one of these classes can also be predicted.
This is made possible by the method `clf_dt.predict_proba()`.

<span style="color:blue"> (c) Run the following code and interpret the input and output.
</span>

In [ ]:
pred_probability = clf_dt.predict_proba(customer)
print("Probability that the customer stays:" , pred_probability[0][0])
print("Probability that the customer leaves:" , pred_probability[0][1])

To see what the decision tree created by the algorithm looks like, the following code can be used.

In [ ]:
from sklearn import tree

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
tree.plot_tree(
    clf_dt,
    ax=ax,
    feature_names=df.drop(['Customer ID', 'LoyaltyID', 'Churn'], axis=1).columns
)
plt.show()

The boxes in the bottom row represent the leaves, the other boxes are the nodes of the tree.
In these boxes, the top row is the condition that is checked.
If the condition is true, the left path is chosen and if the condition is false, the right path is chosen.

- The line with the gini value indicates the distribution of churned customers versus customers who remained with the company at the respective node or leaf.
- The row with the `sample` value indicates how many rows of data (without test data) have reached the respective node or leaf.
- The line with the value value indicates how many passengers (excluding test data) in the respective node or leaf remained as customers (1st value) or churned (2nd value).
- In the leaves, the larger value indicates the prediction the algorithm makes for new data.

<span style="color:blue"> (d) Why are the customers of the test data not counted?
</span> 

<span style="color:blue"> (e) Was this type of decision tree to be expected?
 Are there any distinctions and predictions that are surprising? Why is this?
</span>

### Task 4.3: Calibration of the decision tree
Revise the code given in section 4.3 of the exercise sheet.

<span style="color:blue">
 How can the decision tree be better preset to achieve better accuracy?
 Choose three hyperparameter variants when initializing the decision tree
 (i.e. change the value `max_depth=3`, for example), train the decision tree and determine the score for each of the selected variants.
 What do you notice?
</span>

<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Creative Commons License Agreement" style="border-width:0; display:inline" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a> &nbsp;&nbsp;&nbsp;&nbsp;This work by Marvin Kastner is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Creative Commons Attribution 4.0 International License</a>.